# Setup

All the Imports needed for this experiment.

In [ ]:
from matplotlib import pyplot as plt
from collections import defaultdict
from tqdm import tqdm
import gymnasium as gym
import numpy as np
import pandas as pd
import pickle
import ipywidgets as widgets
from datetime import datetime

import sys
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "env" / "blackjack_env.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from env.blackjack_env import BlackjackEnv
from agents.q_learning_agent import QLearningBlackjackAgent


Define default font size, plot colors etc.

In [ ]:
def setup_plot_style():
    """Definiert das globale Design für alle Matplotlib-Plots."""
    # Schriftgrößen
    plt.rcParams['font.size'] = 11
    plt.rcParams['axes.titlesize'] = 14
    plt.rcParams['axes.titleweight'] = 'bold'
    plt.rcParams['axes.labelsize'] = 12
    plt.rcParams['legend.fontsize'] = 11
    plt.rcParams['xtick.labelsize'] = 10
    plt.rcParams['ytick.labelsize'] = 10
    
    # Linien & Grid
    plt.rcParams['lines.linewidth'] = 2.0
    plt.rcParams['axes.grid'] = True
    plt.rcParams['grid.linestyle'] = '--'
    plt.rcParams['grid.alpha'] = 0.5
    
    # Layout & Render-Qualität
    plt.rcParams['figure.autolayout'] = True
    plt.rcParams['figure.dpi'] = 120
    
    # Optionale Farbpalette (z.B. "Tab10" oder "Set2" für modernen Look)
    plt.style.use('seaborn-v0_8-whitegrid') # Falls ein Grundtheme gewünscht ist

setup_plot_style()
AGENT_STYLES = {    
    "baseline": {"color": "#6b7c93", "label": "Baseline"},
    "counting":  {"color": "#2a9d8f", "label": "Counting"},
}


Create image folder for high-res images and image save function.

In [ ]:
IMAGES_PATH = PROJECT_ROOT / "images"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)
    print(f"Figure saved: {path}")

# Environment & Agents

In [ ]:
def baseline_state_key(obs) -> tuple[int, int, int]:
    player_total, dealer_upcard, usable_ace = obs[:3]

    return (
        int(player_total),
        int(dealer_upcard),
        int(usable_ace)
    )

def counting_state_key(obs: np.ndarray):
    import numpy as np
    player_total, dealer_upcard, usable_ace, running_count, true_count, cards_remaining = obs

    return (
        int(player_total),
        int(dealer_upcard),
        int(usable_ace),

        int(np.clip(running_count, -20, 20)),
        int(np.clip(true_count, -10, 10)),
        int(np.clip(cards_remaining // 52, 0, 6)),
    )


class BaselineBlackjackAgent(QLearningBlackjackAgent):
    def __init__(
            self, 
            env: gym.Env,
            **kwargs):
        
        super().__init__(
            env=env,
            state_encoder=baseline_state_key,
            **kwargs,
        )
        
class CountingBlackjackAgent(QLearningBlackjackAgent):
    def __init__(
            self, 
            env: gym.Env,
            **kwargs):
        
        super().__init__(
            env=env,
            state_encoder=counting_state_key,
            **kwargs,
        )

In [ ]:
SEEDS = [1, 42, 123]
EPISODES_PER_SEED = 10_000_000
EVAL_SEED = 1234
EVAL_EPISODES = 1_000_000
CHECKPOINT_INTERVAL = 5_000_000
SAVE_FINAL_MODELS = True
MODEL_DIR = PROJECT_ROOT / "models"
CHECKPOINT_DIR = MODEL_DIR / "checkpoints"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

agent_config = {
    "learning_rate": 0.01,
    "initial_epsilon": 1.0,
    "final_epsilon": 0.1,
    "epsilon_decay": (1.0 - 0.1) / (EPISODES_PER_SEED * 0.6),
}

def make_env(seed: int, n_episodes: int):
    env = BlackjackEnv(
        num_decks=6,
        penetration=0.75,
        stand_on_soft_17=True,
    )

    env = gym.wrappers.RecordEpisodeStatistics(
        env,
        buffer_length=n_episodes,
    )

    env.reset(seed=seed)
    env.action_space.seed(seed)

    return env

agents = {}

for seed in SEEDS:
    agents[f"baseline-{seed}"] = BaselineBlackjackAgent(
        env=make_env(seed, EPISODES_PER_SEED),
        **agent_config,
    )
    agents[f"counting-{seed}"] = CountingBlackjackAgent(
        env=make_env(seed, EPISODES_PER_SEED),
        **agent_config,
    )

def split_agent_name(name: str) -> tuple[str, int]:
    agent_type, seed = name.rsplit("-", 1)
    return agent_type, int(seed)

def group_agents_by_type(agents):
    grouped = defaultdict(list)

    for name, agent in agents.items():
        agent_type, seed = split_agent_name(name)
        grouped[agent_type].append((seed, agent))

    return grouped


In [ ]:
# Checkpoint-Auswahl fuer Training und Evaluation
NO_ARTIFACT = None
IN_MEMORY_ARTIFACT = "__in_memory__"

def artifact_label(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)

def artifact_options(agent_name: str, include_fresh: bool = False, include_memory: bool = False):
    options = []
    if include_fresh:
        options.append(("Neu trainieren (kein gespeichertes Modell laden)", NO_ARTIFACT))
    if include_memory:
        options.append(("Aktueller Agent im Speicher", IN_MEMORY_ARTIFACT))

    candidates = []
    candidates.extend(MODEL_DIR.glob(f"*{agent_name}*.pkl"))
    candidates.extend((CHECKPOINT_DIR / agent_name).glob("*.pkl"))
    candidates = sorted(
        {path.resolve(): path for path in candidates}.values(),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )

    options.extend((artifact_label(path), str(path)) for path in candidates)
    return options

TRAIN_ARTIFACT_SELECTORS = {
    name: widgets.Dropdown(
        options=artifact_options(name, include_fresh=True),
        value=NO_ARTIFACT,
        description=f"{name}:",
        layout=widgets.Layout(width="760px"),
    )
    for name in agents
}

display(widgets.VBox([widgets.HTML("<b>Training fortsetzen ab:</b>"), *TRAIN_ARTIFACT_SELECTORS.values()]))


# Training

In [ ]:
import time
from datetime import timedelta
from multiprocess import Pool, Manager
import ipywidgets as widgets
from IPython.display import display
from utils.training import train_single_agent

def format_time(seconds):
    """Hilfsfunktion, um Sekunden in ein schönes H:MM:SS Format zu verwandeln."""
    if seconds < 0:
        return "--:--"
    return str(timedelta(seconds=int(seconds)))

if __name__ == "__main__":
    print("Initialisiere erweitertes Progress-Dashboard...")
    
    # 1. Geteiltes Gedächtnis vorbereiten
    manager = Manager()
    progress_dict = manager.dict()
    
    for name in agents.keys():
        progress_dict[name] = 0
        
    # Berechne die absolute Anzahl an Episoden über alle Agenten hinweg
    TOTAL_TARGET_EPISODES = EPISODES_PER_SEED * len(agents)
        
    # 2. NEU: Gesamt-Widgets erstellen (ganz oben platziert)
    overall_bar = widgets.IntProgress(
        value=0, min=0, max=TOTAL_TARGET_EPISODES, 
        description="Gesamtfortschritt:",
        style={'description_width': '140px', 'bar_color': '#28a745'}, # Grün für das große Ziel
        layout=widgets.Layout(width='400px')
    )
    overall_label = widgets.Label(value="Warte auf Start... (Restzeit: --:--:--)")
    overall_row = widgets.HBox([overall_bar, overall_label])
    
    # Eine kleine optische Trennlinie
    separator = widgets.HTML("<hr style='margin: 10px 0; border: 1px solid #ccc;'>")
    
    # 3. Einzelne Agenten-Widgets erstellen
    bars = {}
    labels = {}
    widget_rows = [overall_row, separator] # Gesamt-Balken oben hinzufügen
    
    for name in agents.keys():
        bars[name] = widgets.IntProgress(
            value=0, min=0, max=EPISODES_PER_SEED, 
            description=f"{name}:",
            style={'description_width': '140px'},
            layout=widgets.Layout(width='400px')
        )
        labels[name] = widgets.Label(value="Warte auf Start...")
        widget_rows.append(widgets.HBox([bars[name], labels[name]]))
        
    # Dashboard im Notebook anzeigen
    dashboard = widgets.VBox(widget_rows)
    display(dashboard)
    
    # 4. Aufgaben für Multiprocessing vorbereiten
    tasks = []
    for name, agent in agents.items():
        agent_type, seed = split_agent_name(name)
        selected_artifact = TRAIN_ARTIFACT_SELECTORS[name].value
        tasks.append((name, agent, agent_type, seed, selected_artifact, EPISODES_PER_SEED, CHECKPOINT_INTERVAL, CHECKPOINT_DIR / name, RUN_ID, progress_dict))

    # Zeitmessung starten
    start_time = time.time()
    results = []
    
    with Pool() as pool:
        async_result = pool.starmap_async(train_single_agent, tasks)
        
        # Live-Update-Schleife
        while not async_result.ready():
            elapsed_time = time.time() - start_time
            total_episodes_done = 0
            
            # --- UPDATE EINZELNE AGENTEN ---
            for name in agents.keys():
                current_val = progress_dict.get(name, 0)
                total_episodes_done += current_val
                bars[name].value = current_val
                
                # Prozent & ETA berechnen
                pct = current_val / EPISODES_PER_SEED
                if current_val >= EPISODES_PER_SEED:
                    eta_str = "Fertig!"
                elif pct > 0:
                    total_est_time = elapsed_time / pct
                    remaining_time = max(0.0, total_est_time - elapsed_time)
                    eta_str = f"Restzeit: {format_time(remaining_time)}"
                else:
                    eta_str = "Berechne Restzeit..."
                    
                labels[name].value = f"{pct*100:.1f}% ({current_val:,} / {EPISODES_PER_SEED:,} Ep.) | {eta_str}"
            
            # --- UPDATE GESAMTFORTSCHRITT ---
            overall_bar.value = total_episodes_done
            overall_pct = total_episodes_done / TOTAL_TARGET_EPISODES
            
            if total_episodes_done >= TOTAL_TARGET_EPISODES:
                overall_eta_str = "Fertig!"
            elif overall_pct > 0:
                overall_total_est = elapsed_time / overall_pct
                overall_remaining = max(0.0, overall_total_est - elapsed_time)
                overall_eta_str = f"Gesamt-Restzeit: {format_time(overall_remaining)}"
            else:
                overall_eta_str = "Berechne Gesamt-Restzeit..."
                
            overall_label.value = f"{overall_pct*100:.1f}% ({total_episodes_done:,} / {TOTAL_TARGET_EPISODES:,} Ep.) | {overall_eta_str}"
            
            time.sleep(1.0) # Jede Sekunde aktualisieren
            
        # Am Ende alles sauber auf 100% setzen
        for name in agents.keys():
            bars[name].value = EPISODES_PER_SEED
            labels[name].value = f"100.0% ({EPISODES_PER_SEED:,} Ep.) | Fertig!"
        overall_bar.value = TOTAL_TARGET_EPISODES
        overall_label.value = f"100.0% ({TOTAL_TARGET_EPISODES:,} Ep.) | Fertig!"
            
        # Ergebnisse einsammeln
        results = async_result.get()

    # Ergebnisse zurückschreiben
    for name, trained_agent in results:
        agents[name] = trained_agent

    print(f"\nAlle Trainingsprozesse beendet! Gesamtdauer: {format_time(time.time() - start_time)}")

# Saving

In [ ]:
timestamp = RUN_ID

MODEL_DIR.mkdir(parents=True, exist_ok=True)

if SAVE_FINAL_MODELS:
    for name, agent in agents.items():
        agent_type, seed = split_agent_name(name)
        print(f"Saving {name} agent...")
        
        filename = f"{timestamp}_{name}_agent.pkl"
        path = MODEL_DIR / filename
        
        agent.save(
            path,
            label=f"{name}_{timestamp}",
            artifact_type="final",
            episode=EPISODES_PER_SEED,
            n_episodes=EPISODES_PER_SEED,
            base_seed=seed,
            metadata={
                "agent_name": name,
                "agent_type": agent_type,
                "run_id": RUN_ID,
                "seed": seed,
                "target_episode": EPISODES_PER_SEED,
            },
            include_history=False,
        )
        print(f"-> Erfolgreich gespeichert unter: {path}")
else:
    print("Finales Speichern ist deaktiviert (SAVE_FINAL_MODELS=False).")


# Evaluation

## Greedy Policy Evaluation


In [ ]:
# Auswahl, welches Modell/Checkpoint evaluiert werden soll
EVAL_ARTIFACT_SELECTORS = {
    name: widgets.Dropdown(
        options=artifact_options(name, include_memory=True),
        value=IN_MEMORY_ARTIFACT,
        description=f"{name}:",
        layout=widgets.Layout(width="760px"),
    )
    for name in agents
}

display(widgets.VBox([widgets.HTML("<b>Evaluation-Modell auswaehlen:</b>"), *EVAL_ARTIFACT_SELECTORS.values()]))


In [ ]:
AGENT_CLASSES = {
    "baseline": BaselineBlackjackAgent,
    "counting": CountingBlackjackAgent,
}

def make_agent(agent_name: str, seed: int):
    agent_type, _ = split_agent_name(agent_name)
    return AGENT_CLASSES[agent_type](
        env=make_env(seed, EVAL_EPISODES),
        **agent_config,
    )

def selected_eval_agent(agent_name: str):
    selected_artifact = EVAL_ARTIFACT_SELECTORS[agent_name].value
    if selected_artifact == IN_MEMORY_ARTIFACT:
        return agents[agent_name], "in_memory"

    _, seed = split_agent_name(agent_name)
    eval_agent = make_agent(agent_name, seed=seed)
    artifact = eval_agent.load(selected_artifact)
    label = artifact.get("label") or artifact_label(Path(selected_artifact))
    return eval_agent, label

def greedy_action(agent, obs) -> int:
    state = agent.state_encoder(obs)
    values = agent.q_values.get(state)
    if values is None:
        return 0
    return int(np.argmax(values))

def evaluate_greedy_policy(agent, agent_name: str, n_episodes: int = EVAL_EPISODES, base_seed: int = EVAL_SEED) -> dict:
    env = make_env(base_seed, n_episodes)
    wins = losses = pushes = 0
    action_counts = {0: 0, 1: 0}
    
    sum_rewards = 0.0
    sum_sq_rewards = 0.0

    for episode in tqdm(range(n_episodes), desc=f"eval {agent_name}", leave=False):
        obs, _ = env.reset(seed=base_seed + episode)
        terminated = False
        truncated = False
        episode_reward = 0.0

        while not (terminated or truncated):
            action = greedy_action(agent, obs)
            action_counts[action] = action_counts.get(action, 0) + 1
            obs, reward, terminated, truncated, _ = env.step(action)
            episode_reward += reward

        if episode_reward > 0:
            wins += 1
        elif episode_reward < 0:
            losses += 1
        else:
            pushes += 1

        sum_rewards += episode_reward
        sum_sq_rewards += episode_reward ** 2

    total = wins + losses + pushes
    mean_reward = sum_rewards / total if total else 0.0
    # Varianz-Formel: E[X^2] - E[X]^2
    var_reward = (sum_sq_rewards / total) - (mean_reward ** 2) if total else 0.0
    std_reward = np.sqrt(max(0.0, var_reward))

    return {
        "episodes": total,
        "win_rate": wins / total if total else 0.0,
        "loss_rate": losses / total if total else 0.0,
        "push_rate": pushes / total if total else 0.0,
        "average_reward": mean_reward,
        "std_reward": std_reward,
        "action_distribution": {"stand": action_counts.get(0, 0), "hit": action_counts.get(1, 0)},
    }

greedy_eval_results = {}
for name in agents:
    eval_agent, source_label = selected_eval_agent(name)
    metrics = evaluate_greedy_policy(eval_agent, name)
    greedy_eval_results[name] = {"source": source_label, **metrics}

greedy_eval_df = pd.DataFrame(greedy_eval_results).T
display(greedy_eval_df)


## Training Dynamics

In [ ]:
def get_moving_stats(arr, window: int):
    arr = np.asarray(arr).flatten()
    if len(arr) == 0:
        return arr, arr, np.arange(0)

    min_periods = max(1, window // 10)
    series = pd.Series(arr)

    means = series.rolling(window=window, min_periods=min_periods).mean().to_numpy()
    stds = series.rolling(window=window, min_periods=min_periods).std().to_numpy()
    stds = np.nan_to_num(stds)

    return means, stds, np.arange(len(arr))

def plot_all(agents: dict, reward_window: int = 1000, td_window: int = 5000):

    fig, axs = plt.subplots(1, 2, figsize=(16, 6))

    grouped = group_agents_by_type(agents)

    for agent_type, agent_list in grouped.items():

        style = AGENT_STYLES[agent_type]

        # ======================
        # REWARDS (correct)
        # ======================

        reward_curves = []

        for seed, agent in agent_list:

            rewards = np.asarray(agent.episode_rewards)
            means, _, _ = get_moving_stats(rewards, reward_window)

            reward_curves.append(means)

        # align lengths (IMPORTANT)
        min_len = min(len(c) for c in reward_curves)
        reward_curves = np.array([c[:min_len] for c in reward_curves])

        reward_mean = reward_curves.mean(axis=0)
        reward_std = reward_curves.std(axis=0)

        x_r = np.arange(len(reward_mean))

        axs[0].plot(
            x_r,
            reward_mean,
            label=style["label"],
            color=style["color"],
            linewidth=2.5,
        )

        axs[0].fill_between(
            x_r,
            reward_mean - reward_std,
            reward_mean + reward_std,
            color=style["color"],
            alpha=0.2,
        )

        # ======================
        # TD ERROR (correct)
        # ======================

        td_curves = []

        for seed, agent in agent_list:

            td_errors = np.asarray(agent.training_error)
            means, _, _ = get_moving_stats(td_errors, td_window)

            td_curves.append(means)

        # align lengths (IMPORTANT)
        min_len = min(len(c) for c in td_curves)
        td_curves = np.array([c[:min_len] for c in td_curves])

        td_mean = td_curves.mean(axis=0)
        td_std = td_curves.std(axis=0)

        x_t = np.arange(len(td_mean))

        axs[1].plot(
            x_t,
            td_mean,
            label=style["label"],
            color=style["color"],
            linewidth=2.5,
        )

        axs[1].fill_between(
            x_t,
            td_mean - td_std,
            td_mean + td_std,
            color=style["color"],
            alpha=0.2,
        )

    axs[0].set_title("Episode Rewards (Mittelwert über Seeds)")
    axs[0].set_xlabel("Episode")
    axs[0].set_ylabel("Reward")
    axs[0].legend()

    axs[1].set_title("TD Error (Mittelwert über Seeds)")
    axs[1].set_xlabel("Episode")
    axs[1].set_ylabel("TD Error")
    axs[1].legend()

    save_fig("training_results_combined")
    plt.show()

In [ ]:
plot_all(agents, reward_window=1000, td_window=5000)

## Performance Comparison

In [ ]:
print("Starte Trainings-Auswertung der Agenten...\n")

def get_reward_matrix(agent_list):
    return np.vstack([
        np.asarray(agent.episode_rewards)
        for seed, agent in agent_list
    ])


def flatten_rewards(agent_list):
    return get_reward_matrix(agent_list).ravel()


def calculate_metrics(rewards: np.ndarray) -> dict:
    rewards = np.asarray(rewards)

    wins = np.sum(rewards == 1)
    losses = np.sum(rewards == -1)
    pushes = np.sum(rewards == 0)
    total = len(rewards)

    return {
        "win_rate": wins / total if total > 0 else 0.0,
        "loss_rate": losses / total if total > 0 else 0.0,
        "push_rate": pushes / total if total > 0 else 0.0,
        "average_reward": np.mean(rewards) if total > 0 else 0.0,
        "std_reward": np.std(rewards) if total > 0 else 0.0,
    }


grouped = group_agents_by_type(agents)

per_seed_rows = []
for agent_type, agent_list in grouped.items():
    for seed, agent in agent_list:
        row = calculate_metrics(np.asarray(agent.episode_rewards))
        row.update({"agent_type": agent_type, "seed": seed})
        per_seed_rows.append(row)

per_seed_metrics_df = pd.DataFrame(per_seed_rows).sort_values(["agent_type", "seed"])
display(per_seed_metrics_df)

baseline_rewards = flatten_rewards(grouped["baseline"])
counting_rewards = flatten_rewards(grouped["counting"])

baseline_metrics = calculate_metrics(baseline_rewards)
counting_metrics = calculate_metrics(counting_rewards)

summary_metrics_df = pd.DataFrame({
    "baseline": baseline_metrics,
    "counting": counting_metrics,
}).T
display(summary_metrics_df)

for name, metrics in [("Baseline", baseline_metrics), ("Counting", counting_metrics)]:
    print(f"{name} metrics:")
    print(f"  Win Rate:       {metrics['win_rate']:.1%}")
    print(f"  Loss Rate:      {metrics['loss_rate']:.1%}")
    print(f"  Push Rate:      {metrics['push_rate']:.1%}")
    print(f"  Avg Reward:     {metrics['average_reward']:.3f}")
    print(f"  Std Reward:     {metrics['std_reward']:.3f}")
    print()


In [ ]:
if "greedy_eval_df" not in globals():
    raise RuntimeError("Bitte zuerst die Greedy-Evaluation ausführen.")

from utils.evaluation import calculate_evaluation_cis, plot_evaluation_metrics_with_cis

print("Zusammenfassung pro Seed:")
display(greedy_eval_df[["source", "episodes", "win_rate", "loss_rate", "push_rate", "average_reward"]])

print("\nBerechne mathematische 95% Konfidenzintervalle:")
display(calculate_evaluation_cis(greedy_eval_df))

print("\nGeneriere 2x2 Signifikanz-Plots...")
plot_evaluation_metrics_with_cis(
    greedy_eval_df=greedy_eval_df,
    agent_styles=AGENT_STYLES,
    split_agent_name_func=split_agent_name,
    save_fig_func=save_fig
)

# Policy & Q-Value Visualization
Hier analysieren wir interaktiv die gelernten Strategien der Agenten und vergleichen sie mit der Basic Strategy.

In [ ]:
import ipywidgets as widgets
from utils.visualizations import plot_policy_and_q_values

def interactive_plot_wrapper(agent_name, true_count):
    if agent_name in agents:
        plot_policy_and_q_values(
            agent=agents[agent_name], 
            agent_name=agent_name, 
            split_agent_name_func=split_agent_name, 
            save_fig_func=save_fig, 
            true_count=true_count
        )

# --- STEUERUNG FÜR AGENT A (LINKS) ---
agent_selector_a = widgets.Dropdown(options=list(agents.keys()), value=list(agents.keys())[0], description='Agent A:')
true_count_slider_a = widgets.IntSlider(value=0, min=-10, max=10, step=1, description='True Count A:', disabled=True)

def update_slider_visibility_a(*args):
    agent_type, _ = split_agent_name(agent_selector_a.value)
    true_count_slider_a.disabled = (agent_type == "baseline")

agent_selector_a.observe(update_slider_visibility_a, 'value')
plot_a = widgets.interactive(interactive_plot_wrapper, agent_name=agent_selector_a, true_count=true_count_slider_a)


# --- STEUERUNG FÜR AGENT B (RECHTS) ---
# Startwert auf den letzten Eintrag gesetzt, damit direkt zwei unterschiedliche Agenten laden
agent_selector_b = widgets.Dropdown(options=list(agents.keys()), value=list(agents.keys())[-1], description='Agent B:')
true_count_slider_b = widgets.IntSlider(value=0, min=-10, max=10, step=1, description='True Count B:', disabled=True)

def update_slider_visibility_b(*args):
    agent_type, _ = split_agent_name(agent_selector_b.value)
    true_count_slider_b.disabled = (agent_type == "baseline")

agent_selector_b.observe(update_slider_visibility_b, 'value')
plot_b = widgets.interactive(interactive_plot_wrapper, agent_name=agent_selector_b, true_count=true_count_slider_b)

update_slider_visibility_a()
update_slider_visibility_b()

widgets.HBox([plot_a, plot_b])